In [6]:
import os
print("Current working directory:", os.getcwd())


Current working directory: c:\Users\USER\Documents\FloodRisk-Ibadan-DeepLearning\experiments


In [7]:
%pwd

'c:\\Users\\USER\\Documents\\FloodRisk-Ibadan-DeepLearning\\experiments'

In [8]:
os.chdir("../")

In [9]:
print("Current working directory:", os.getcwd())

Current working directory: c:\Users\USER\Documents\FloodRisk-Ibadan-DeepLearning


In [14]:
import os
import numpy as np
import rasterio
from rasterio.windows import Window
from sklearn.model_selection import train_test_split

# ==== CONFIGURATION ====
INPUT_STACK = r"data\Ibadan_Flood_Project\Sentinel_Stacks\ibadan_2023_dry_stack.tif"   # example Sentinel-2 stack
INPUT_MASK  = r"data\Ibadan_Flood_Project\Masks2\ibadan_2023_dry_label.tif"            # corresponding mask
PATCH_SIZE = 256                                              # tile size (256x256)
OUTPUT_DIR = "dataset/tiles/"

# Create output folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, "masks"), exist_ok=True)

# ==== STEP 1: Load data ====
with rasterio.open(INPUT_STACK) as src_img, rasterio.open(INPUT_MASK) as src_mask:
    img_array = src_img.read()   # shape: (bands, H, W)
    mask_array = src_mask.read(1)  # single band (H, W)

print("Image shape:", img_array.shape)  # e.g. (6 bands, 3000, 3000)
print("Mask shape:", mask_array.shape)

# ==== STEP 2: Extract patches ====
patches_img = []
patches_mask = []

height, width = mask_array.shape

for i in range(0, height, PATCH_SIZE):
    for j in range(0, width, PATCH_SIZE):
        # Crop window
        img_patch = img_array[:, i:i+PATCH_SIZE, j:j+PATCH_SIZE]
        mask_patch = mask_array[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
        
        # Skip if patch is smaller than desired size (edge tiles)
        if img_patch.shape[1] != PATCH_SIZE or img_patch.shape[2] != PATCH_SIZE:
            continue
        
        # Skip patches with no water at all (optional, reduces imbalance)
        if mask_patch.sum() == 0:
            continue
        
        patches_img.append(img_patch)
        patches_mask.append(mask_patch)

print(f"Total patches extracted: {len(patches_img)}")

# ==== STEP 3: Split train/val/test ====
train_idx, test_idx = train_test_split(range(len(patches_img)), test_size=0.2, random_state=42)
train_idx, val_idx  = train_test_split(train_idx, test_size=0.1, random_state=42)

def save_patches(indices, split):
    for k, idx in enumerate(indices):
        np.save(os.path.join(OUTPUT_DIR, split, "images", f"img_{k}.npy"), patches_img[idx])
        np.save(os.path.join(OUTPUT_DIR, split, "masks", f"mask_{k}.npy"), patches_mask[idx])

save_patches(train_idx, "train")
save_patches(val_idx, "val")
save_patches(test_idx, "test")

print("✅ Patches saved into dataset/tiles/ folder")


Image shape: (6, 1251, 1134)
Mask shape: (1251, 1134)
Total patches extracted: 15
✅ Patches saved into dataset/tiles/ folder
